# CHiRPE Tutorial: Transcript-Level Voting Inside ONNX

This notebook shows how to add transcript-level aggregation to a classifier-only CHiRPE ONNX model.

The key contract is:

```text
one ONNX request = all segment tensors for one transcript
```

The augmented ONNX model keeps the original segment-level `logits` output and adds:

- `segment_probabilities`
- `segment_labels`
- `transcript_probabilities`
- `transcript_label_average`
- `transcript_label_majority`


## What This Does

The original classifier ONNX model maps a batch of segment token tensors to segment logits:

```text
input_ids [num_segments, sequence_length]
attention_mask [num_segments, sequence_length]
token_type_ids [num_segments, sequence_length]  # when required by the backbone
  -> logits [num_segments, 2]
```

The voting-augmented model appends ONNX ops after `logits`:

```text
logits
  -> Softmax -> segment_probabilities
  -> ArgMax  -> segment_labels
  -> ReduceMean(segment_probabilities, axis=0) -> transcript_probabilities
  -> ArgMax(transcript_probabilities) -> transcript_label_average
  -> binary majority vote(segment_labels) -> transcript_label_majority
```


## Important Limitations

This tutorial is for **classifier-only ONNX** models where segment summaries have already been tokenized outside ONNX.

It does not aggregate across separate one-segment requests. If each segment is sent as a separate ONNX or Triton request, the model has no memory of previous requests and transcript-level aggregation must remain outside ONNX.

The original fused string-input ONNX path in this repo is different: it is built around a tokenizer custom op path that handles one segment string per request. For tokenizer-inside-ONNX transcript voting, use the fixed-slot builder `scripts/onnx/build_fused_string_voting_onnx.py`, which accepts `text[15]` plus `segment_mask[15]`.


In [ ]:
from pathlib import Path
import json
import sys

import numpy as np

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "src").exists():
    raise RuntimeError("Run this notebook from the repository root or notebooks/ directory.")

sys.path.insert(0, str(REPO_ROOT / "src"))
sys.path.insert(0, str(REPO_ROOT))

print(f"Repository root: {REPO_ROOT}")

In [ ]:
import onnx
import onnxruntime as ort
from transformers import AutoTokenizer

from chirpe.data.preprocessor import TranscriptPreprocessor
from scripts.onnx.add_transcript_voting_onnx import (
    SEGMENT_LABELS,
    SEGMENT_PROBABILITIES,
    TRANSCRIPT_LABEL_AVERAGE,
    TRANSCRIPT_LABEL_MAJORITY,
    TRANSCRIPT_PROBABILITIES,
    add_transcript_voting_outputs,
    metadata_payload,
)

## Choose Source Artifacts

This tutorial uses the existing BERT classifier-only ONNX export if available. It also needs the matching Hugging Face checkpoint directory only for tokenization in the notebook.


In [ ]:
BACKBONE = "bert"

SOURCE_ONNX_CANDIDATES = [
    REPO_ROOT / "outputs/string_onnx_baseline_classifier/chirpe_bert_baseline/1/model.onnx",
    REPO_ROOT / "outputs/triton_model_repository/chirpe_classifier/1/model.onnx",
]
CHECKPOINT_CANDIDATES = [
    REPO_ROOT / "outputs/string_onnx_checkpoints/bert/best_model",
    REPO_ROOT / "outputs/test-config-save/best_model",
]
DATA_FILE = REPO_ROOT / "data/synthetic/test.json"
OUTPUT_DIR = REPO_ROOT / "outputs/notebook_transcript_voting_onnx" / BACKBONE
OUTPUT_MODEL = OUTPUT_DIR / "model.onnx"
OUTPUT_METADATA = OUTPUT_DIR / "voting_metadata.json"

SOURCE_ONNX = next((path for path in SOURCE_ONNX_CANDIDATES if path.exists()), SOURCE_ONNX_CANDIDATES[0])
CHECKPOINT_DIR = next((path for path in CHECKPOINT_CANDIDATES if path.exists()), CHECKPOINT_CANDIDATES[0])

for path, label in [(SOURCE_ONNX, "source ONNX"), (CHECKPOINT_DIR, "checkpoint"), (DATA_FILE, "test data")]:
    if not path.exists():
        raise FileNotFoundError(f"Missing {label}: {path}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Source classifier ONNX: {SOURCE_ONNX.relative_to(REPO_ROOT)}")
print(f"Tokenizer checkpoint: {CHECKPOINT_DIR.relative_to(REPO_ROOT)}")
print(f"Output ONNX: {OUTPUT_MODEL.relative_to(REPO_ROOT)}")

## Build the Voting-Augmented ONNX Model

The helper below loads the existing classifier ONNX graph and appends pure ONNX post-processing nodes. The original classifier weights and tokenizer behavior are not changed.


In [ ]:
model = onnx.load(str(SOURCE_ONNX))
model = add_transcript_voting_outputs(model, logits_output_name="logits")
onnx.save(model, str(OUTPUT_MODEL))

with open(OUTPUT_METADATA, "w") as file:
    json.dump(
        metadata_payload(
            input_model=SOURCE_ONNX,
            output_model=OUTPUT_MODEL,
            logits_output_name="logits",
        ),
        file,
        indent=2,
    )

print(f"Wrote: {OUTPUT_MODEL.relative_to(REPO_ROOT)}")
print(f"Wrote: {OUTPUT_METADATA.relative_to(REPO_ROOT)}")

In [ ]:
session = ort.InferenceSession(str(OUTPUT_MODEL), providers=["CPUExecutionProvider"])

print("Inputs:")
for item in session.get_inputs():
    print({"name": item.name, "type": item.type, "shape": item.shape})

print("\nOutputs:")
for item in session.get_outputs():
    print({"name": item.name, "type": item.type, "shape": item.shape})

## Prepare One Transcript's Segment Batch

Preprocessing still happens outside ONNX. We load one synthetic transcript, segment it with CHiRPE, summarize each segment, and tokenize all segment summaries as one batch.


In [ ]:
with open(DATA_FILE, "r") as file:
    test_items = json.load(file)

sample_item = test_items[0]
print(json.dumps({k: sample_item[k] for k in ["participant_id", "label"]}, indent=2))
print(f"Transcript utterances: {len(sample_item['transcript'])}")

In [ ]:
preprocessor = TranscriptPreprocessor(
    segmentation_threshold=0.80,
    use_llm_summarizer=False,
)
processed = preprocessor.process_transcript(
    sample_item["transcript"],
    participant_id=sample_item["participant_id"],
)

segments = processed["segments"][:3]
if not segments:
    raise RuntimeError("No segments were extracted from the sample transcript.")

print(f"Using {len(segments)} segments for one transcript-level ONNX request.")
for index, segment in enumerate(segments, start=1):
    print(f"\nSegment {index}: {segment['domain']}")
    print(segment["summary"][:300])

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT_DIR)
summaries = [segment["summary"] for segment in segments]

encoded = tokenizer(
    summaries,
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="np",
)

input_names = [item.name for item in session.get_inputs()]
ort_inputs = {name: encoded[name].astype(np.int64) for name in input_names if name in encoded}
missing_inputs = sorted(set(input_names) - set(ort_inputs))
if missing_inputs:
    raise RuntimeError(f"Tokenizer did not produce required ONNX inputs: {missing_inputs}")

for name, value in ort_inputs.items():
    print(name, value.shape, value.dtype)

## Run ONNX Inference With Transcript Voting

This is a single ONNX Runtime call. The batch dimension is the transcript's segment count.


In [ ]:
output_names = [
    "logits",
    SEGMENT_PROBABILITIES,
    SEGMENT_LABELS,
    TRANSCRIPT_PROBABILITIES,
    TRANSCRIPT_LABEL_AVERAGE,
    TRANSCRIPT_LABEL_MAJORITY,
]

outputs = session.run(output_names, ort_inputs)
(
    logits,
    segment_probabilities,
    segment_labels,
    transcript_probabilities,
    transcript_label_average,
    transcript_label_majority,
) = outputs

label_map = {0: "Healthy", 1: "CHR-P"}

print("Segment labels:", segment_labels.tolist())
print("Segment probabilities:")
print(segment_probabilities)
print("\nTranscript probabilities:", transcript_probabilities.tolist())
print("Average-vote label:", int(transcript_label_average[0]), label_map[int(transcript_label_average[0])])
print("Majority-vote label:", int(transcript_label_majority[0]), label_map[int(transcript_label_majority[0])])

## Validate Against NumPy Reference

The ONNX post-processing should match the same operations computed in Python.


In [ ]:
exp_logits = np.exp(logits - logits.max(axis=-1, keepdims=True))
np_segment_probabilities = exp_logits / exp_logits.sum(axis=-1, keepdims=True)
np_segment_labels = np_segment_probabilities.argmax(axis=-1).astype(np.int64)
np_transcript_probabilities = np_segment_probabilities.mean(axis=0)
np_average_label = int(np_transcript_probabilities.argmax())
np_majority_label = int(np.bincount(np_segment_labels, minlength=2).argmax())

checks = {
    "segment_probabilities": np.allclose(segment_probabilities, np_segment_probabilities, atol=1e-6),
    "segment_labels": np.array_equal(segment_labels, np_segment_labels),
    "transcript_probabilities": np.allclose(transcript_probabilities, np_transcript_probabilities, atol=1e-6),
    "average_label": int(transcript_label_average[0]) == np_average_label,
    "majority_label": int(transcript_label_majority[0]) == np_majority_label,
}

print(json.dumps(checks, indent=2))
assert all(checks.values())

## What To Serve With Triton

If you serve this augmented model with Triton, update `config.pbtxt` to include the extra outputs. For classifier-only BERT exports, the output section should include tensors like:

```text
output [
  { name: "logits" data_type: TYPE_FP32 dims: [ 2 ] },
  { name: "segment_probabilities" data_type: TYPE_FP32 dims: [ 2 ] },
  { name: "segment_labels" data_type: TYPE_INT64 dims: [ ] },
  { name: "transcript_probabilities" data_type: TYPE_FP32 dims: [ 2 ] },
  { name: "transcript_label_average" data_type: TYPE_INT64 dims: [ 1 ] },
  { name: "transcript_label_majority" data_type: TYPE_INT64 dims: [ 1 ] }
]
```

With `max_batch_size > 0`, Triton treats the first dimension as its own request batch dimension. For transcript-level aggregation, the practical contract is usually simpler with `max_batch_size: 0`, where the ONNX first dimension directly represents `num_segments`.


## Summary

This notebook demonstrated how to:

1. Load a classifier-only CHiRPE ONNX model.
2. Append transcript-level voting outputs inside the ONNX graph.
3. Preprocess one raw transcript outside ONNX.
4. Send all segment token tensors as one ONNX request.
5. Read both segment-level and transcript-level outputs from ONNX Runtime.
6. Verify the ONNX aggregation against NumPy reference logic.
